In [16]:
### Cell 1 – Compute SSIM Similarity Scores

# 1) install scikit-image if needed
!pip install --quiet scikit-image

import numpy as np
from skimage.metrics import structural_similarity as ssim
from PIL import Image

# load reference image as numpy float [0,1]
ref_img = Image.open(os.path.join(DATA_DIR, "original2.png")).convert("L")
ref_np  = np.array(ref_img, dtype=np.float32) / 255.0

scores = []
for img_path in dataset.files:
    # load each child drawing
    img = Image.open(img_path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0

    # ensure same shape
    if arr.shape != ref_np.shape:
        arr = np.array(img.resize(ref_img.size), dtype=np.float32) / 255.0

    # compute SSIM (value in [-1,1], with 1 = perfect match)
    score = ssim(ref_np, arr, data_range=1.0)
    scores.append((img_path, score))

# sort by decreasing SSIM (highest = most similar)
scores.sort(key=lambda x: x[1], reverse=True)
print("Top 5 most similar by SSIM:", scores[:5])


Top 5 most similar by SSIM: [('/content/shape2/original2.png', np.float64(1.0)), ('/content/shape2/10731.png', np.float64(0.9000936691818939)), ('/content/shape2/10548.png', np.float64(0.8958081605544788)), ('/content/shape2/10553.png', np.float64(0.8941834948643649)), ('/content/shape2/10659.png', np.float64(0.8914421823251624))]


In [17]:
### Cell 2 – Generate Self-Contained HTML Ranking by SSIM

import base64

html = [
    "<!DOCTYPE html>",
    "<html><head><meta charset='utf-8'><title>SSIM Ranking</title></head><body>",
    "<h1>Shape SSIM Ranking (to original2.png)</h1>",
    "<ol style='list-style:none;'>"
]

for path, score in scores:
    ext  = path.rsplit(".",1)[-1].lower()
    mime = "image/png" if ext=="png" else "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    html.append(
        f"<li style='margin-bottom:20px;'>"
        f"<img src='data:{mime};base64,{b64}' width='200'><br>"
        f"<strong>SSIM:</strong> {score:.4f}"
        f"</li>"
    )

html += ["</ol></body></html>"]

out_file = "/content/ssim_ranking.html"
with open(out_file, "w") as f:
    f.write("\n".join(html))

print("Wrote SSIM‐based ranking to", out_file)


Wrote SSIM‐based ranking to /content/ssim_ranking.html
